In [2]:
!pip install ragas datasets

  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 995.3/995.3 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 28.0 MB/s eta 0:00:00
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
  Attempting uninstall: jinja2
    Found existing installation: Jinja2 3.1.2
    Uninstalling Jinja2-3.1.2:
      Successfully uninstalled Jinja2-3.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires packaging<26,>=20, but you have packaging 26.0 which is incompatible.
jupyter-server 2.0.1 requires anyio<4,>=3.1.0, but you have anyio 4.13.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
import os
import pandas as pd
from datasets import Dataset
from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from ragas import evaluate

# FIX: Import the capitalized Class versions of the metrics
from ragas.metrics import Faithfulness, ContextPrecision 

# Load API Key
load_dotenv()

print("1. Loading Local RAG Architecture (BGE-Large)...")
rag_embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5", model_kwargs={'device': 'mps'})
vector_store = FAISS.load_local("../data/faiss_index", rag_embeddings, allow_dangerous_deserialization=True)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
rag_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

print("2. Configuring RAGAS Judge Models...")
judge_llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
judge_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# 3. Define the Test Suite
questions = [
    "What are the common Initial Access techniques shared by Akira Ransomware and Scattered Spider?",
    "Which ransomware group explicitly exploited the Citrix Bleed vulnerability (CVE-2023-4966)?",
    "Describe the living-off-the-land techniques used by Volt Typhoon to maintain persistence."
]

references = [
    "Both Akira Ransomware and Scattered Spider utilize Phishing (T1566) for initial access.",
    "LockBit 3.0 ransomware (and its affiliates) explicitly exploited the Citrix Bleed vulnerability.",
    "Volt Typhoon uses built-in tools like wmic, ntdsutil, netsh, and PowerShell for living-off-the-land persistence."
]

data_samples = {
    "question": [],
    "answer": [],
    "contexts": [],
    "reference": [] 
}

print("3. Generating Answers for Evaluation...")
for i, query in enumerate(questions):
    print(f"   -> Answering: {query}")
    docs = retriever.invoke(query)
    context_list = [doc.page_content for doc in docs]
    
    prompt = f"Context: {' '.join(context_list)}\n\nQuestion: {query}\nAnswer based ONLY on context."
    answer = rag_llm.invoke(prompt).content
    
    data_samples["question"].append(query)
    data_samples["answer"].append(answer)
    data_samples["contexts"].append(context_list)
    data_samples["reference"].append(references[i]) 

dataset = Dataset.from_dict(data_samples)

print("4. Executing Mathematical RAGAS Evaluation... (This takes a moment)")
# FIX: Instantiate the metric classes using parentheses ()
score = evaluate(
    dataset,
    metrics=[Faithfulness(), ContextPrecision()],
    llm=judge_llm,
    embeddings=judge_embeddings
)

df_results = score.to_pandas()
df_results.to_csv("../results/final_ragas_metrics.csv", index=False)
print("\nEvaluation Complete! Results saved to results/final_ragas_metrics.csv")
display(df_results)

/var/folders/1k/v08vylqd303d4r8_1w5k5p100000gn/T/ipykernel_43196/146478037.py:11: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, ContextPrecision
/var/folders/1k/v08vylqd303d4r8_1w5k5p100000gn/T/ipykernel_43196/146478037.py:11: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import Faithfulness, ContextPrecision


1. Loading Local RAG Architecture (BGE-Large)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2. Configuring RAGAS Judge Models...
3. Generating Answers for Evaluation...
   -> Answering: What are the common Initial Access techniques shared by Akira Ransomware and Scattered Spider?
   -> Answering: Which ransomware group explicitly exploited the Citrix Bleed vulnerability (CVE-2023-4966)?
   -> Answering: Describe the living-off-the-land techniques used by Volt Typhoon to maintain persistence.
4. Executing Mathematical RAGAS Evaluation... (This takes a moment)


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

Exception raised in Job[1]: TimeoutError()



Evaluation Complete! Results saved to results/final_ragas_metrics.csv


,user_input,retrieved_contexts,response,reference,faithfulness,context_precision
0,What are the common Initial Access techniques ...,[|Technique Title|ID|Use|\n|---|---|---|\n|Pro...,"Based on the provided context, there are no co...",Both Akira Ransomware and Scattered Spider uti...,0.875,NaN
1,Which ransomware group explicitly exploited th...,[**==> picture [169 x 71] intentionally omitte...,LockBit 3.0 ransomware affiliates explicitly e...,LockBit 3.0 ransomware (and its affiliates) ex...,1.000,1.0
2,Describe the living-off-the-land techniques us...,[Page 2 of 45 | Product ID: AA24-038A \n\n**...,"Based on the context provided, Volt Typhoon ac...","Volt Typhoon uses built-in tools like wmic, nt...",1.000,0.0
